# 你的第一个 Agent

第一个 Agent 只做一件事：让模型决定是否执行 Bash 命令，并把命令执行结果反馈给模型。

本节参考 [learn-claude-code/s01_agent_loop](https://github.com/shareAI-lab/learn-claude-code/tree/main/s01_agent_loop)。原项目使用 Claude Code 的教学示例，本课程只做少量调整：读取本节目录下的 `.env`，并使用 Claude API 兼容格式连接 GLM 后端。

## 目标

完成本节后，你应该理解：

- Agent 和普通聊天应用的区别；
- 为什么只用一个 `bash` 工具也能形成 Agent；
- Anthropic Messages API 中 `text`、`tool_use`、`tool_result` 这几类 block 的含义；
- `while True` 循环如何把“模型决策”和“工具执行”接起来。

## 问题：聊天应用不会自己执行命令

如果直接问大模型“当前文件夹有哪些文件”，模型可能会回答：

```bash
ls
```

但普通聊天应用到这里就停了。它不会真的执行 `ls`，也看不到命令输出。

你可以手动复制命令到终端运行，再把结果粘贴回对话框。Agent 要做的事情，就是把这个“手动中转”自动化。

## 解决方案：一个循环和一个 Bash 工具

最小 Agent 的核心流程是：

```text
模型请求使用工具 -> 程序执行工具 -> 工具结果反馈给模型 -> 模型继续判断
```

![Agent Loop](figures/agent-loop.svg)

图示来源：[learn-claude-code/s01_agent_loop](https://github.com/shareAI-lab/learn-claude-code/tree/main/s01_agent_loop)。

循环只需要关注 `stop_reason`：

| 信号 | 含义 | 程序动作 |
| --- | --- | --- |
| `tool_use` | 模型要调用工具 | 执行工具，把结果放回上下文 |
| 其他情况 | 模型不再调用工具 | 结束循环，输出最终回答 |

## 1. 读取配置并创建客户端

先在 `books/06_Agent/.env` 中保存接口地址、API Key 和模型名称：

```text
ANTHROPIC_BASE_URL=https://open.bigmodel.cn/api/anthropic
ANTHROPIC_AUTH_TOKEN=你的 Key 填写在这里
MODEL_ID=glm-5.2
```

下面的代码会读取 `.env`，并创建 Anthropic 兼容客户端。

In [1]:
import json
import os
import subprocess
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv

# 兼容两种运行位置：项目根目录，或 books/06_Agent 目录。
NOTEBOOK_DIR = Path("books/06_Agent") if Path("books/06_Agent").exists() else Path.cwd()
load_dotenv(NOTEBOOK_DIR / ".env", override=True)
load_dotenv(override=True)

if not os.getenv("ANTHROPIC_AUTH_TOKEN"):
    raise RuntimeError("请先在 books/06_Agent/.env 中设置 ANTHROPIC_AUTH_TOKEN")
if not os.getenv("MODEL_ID"):
    raise RuntimeError("请先在 books/06_Agent/.env 中设置 MODEL_ID")

client = Anthropic(
    auth_token=os.getenv("ANTHROPIC_AUTH_TOKEN"),
    base_url=os.getenv("ANTHROPIC_BASE_URL"),
)
MODEL = os.environ["MODEL_ID"]
MODEL

'glm-5.2'

这里使用的是 Anthropic SDK，但重点不是 SDK 名字，而是 Claude API 兼容的消息格式和工具调用协议。模型可以返回 `tool_use`，程序执行工具后，再把结果作为 `tool_result` 放回去。

## 2. 定义系统提示词

系统提示词告诉模型：你现在不是普通聊天助手，而是可以使用 Bash 的 Coding Agent。

In [2]:
SYSTEM = f"You are a coding agent at {NOTEBOOK_DIR.resolve()}. Use bash to solve tasks. Act, don't explain."
SYSTEM

"You are a coding agent at /Users/sunq/suibe_courses/modern_programming/books/06_Agent. Use bash to solve tasks. Act, don't explain."

这句话包含三个约束：

- 当前工作目录是哪里；
- 解决问题时可以使用 Bash；
- 优先行动，而不是只解释。

## 3. 工具：Bash



### Bash工具定义
第一个 Agent 只有一个工具：`bash`。工具定义会被发给模型，让模型知道工具名称、用途和参数格式。

In [3]:
TOOLS = [{
    "name": "bash",
    "description": "Run a shell command.",
    "input_schema": {
        "type": "object",
        "properties": {"command": {"type": "string"}},
        "required": ["command"],
    },
}]

TOOLS

[{'name': 'bash',
  'description': 'Run a shell command.',
  'input_schema': {'type': 'object',
   'properties': {'command': {'type': 'string'}},
   'required': ['command']}}]

这段定义的含义是：如果模型想执行命令，就应该生成一个 `bash` 工具调用，并在参数中提供 `command` 字符串。

### Bash工具实现

`run_bash` 是真正接触操作系统的地方。为了课堂演示，它做了简单安全限制：拦截明显危险命令、设置超时，并收集标准输出和错误输出。

In [4]:
def run_bash(command: str) -> str:
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        result = subprocess.run(
            command,
            shell=True,
            cwd=NOTEBOOK_DIR,
            capture_output=True,
            text=True,
            timeout=120,
        )
        output = (result.stdout + result.stderr).strip()
        return output[:50000] if output else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"
    except (FileNotFoundError, OSError) as exc:
        return f"Error: {exc}"

先用一个安全命令验证工具能运行：

In [5]:
run_bash("echo 'hello'")

'hello'

## 4. Anthropic API 与 block 类型

### 消息请求
Anthropic Messages API 的请求核心如下：

In [6]:
demo_messages = [{"role": "user", "content": "创建 HelloWord.py，然后运行"}]

demo_request = {
    "model": MODEL,
    "system": SYSTEM,
    "messages": demo_messages,
    "tools": TOOLS,
    "max_tokens": 8000,
}

demo_request.keys()

dict_keys(['model', 'system', 'messages', 'tools', 'max_tokens'])

这些字段的含义是：

| 字段 | 作用 |
| --- | --- |
| `model` | 使用哪个模型，例如 `glm-5.2` |
| `system` | 系统提示词，定义 Agent 的角色和行为约束 |
| `messages` | 对话历史，包含用户消息、助手消息和工具结果 |
| `tools` | 可供模型调用的工具定义 |
| `max_tokens` | 限制模型本次最多生成多少 token |



### 消息响应与Block
一个重要点是：`response.content` 不是普通字符串，而是 block 列表。每个 block 都有自己的 `type`。

常见 block 类型如下：

| block 类型 | 方向 | 含义 |
| --- | --- | --- |
| `text` | 模型返回 | 普通文本回答 |
| `tool_use` | 模型返回 | 模型请求调用某个工具 |
| `tool_result` | 程序回传 | 程序执行工具后，把结果返回给模型 |

下面真实调用两次模型，并打印返回的 block：

- 第一个问题需要查看文件夹，通常会返回 `tool_use`；
- 第二个问题是概念解释，通常只返回 `text`。

In [7]:
def print_response_blocks(title: str, response):
    print(f"\n## {title}")
    print("stop_reason:", response.stop_reason)
    for block in response.content:
        print("type:", block.type)
        if block.type == "text":
            print("text:", block.text)
        elif block.type == "tool_use":
            print("tool:", block.name)
            print("input:", block.input)
            print("id:", block.id)
        print("---")



cnn_messages = [{"role": "user", "content": "一句话解释卷积神经网络？"}]
cnn_response = client.messages.create(
    model=MODEL,
    system=SYSTEM,
    messages=cnn_messages,
    tools=TOOLS,
    max_tokens=8000,
)
print_response_blocks("概念问题：卷积神经网络", cnn_response)

py_messages = [{"role": "user", "content": "创建 HelloWord.py，然后运行"}]
py_response = client.messages.create(
    model=MODEL,
    system=SYSTEM,
    messages=py_messages,
    tools=TOOLS,
    max_tokens=8000,
)
print_response_blocks("创建 HelloWord.py，然后运行", py_response)


## 概念问题：卷积神经网络
stop_reason: end_turn
type: text
text: **卷积神经网络（CNN）**是一种通过"卷积核"自动提取图像局部特征（如边缘、纹理、形状）并逐层组合成高层语义（如物体类别）的深度学习模型，本质上是用"滑动窗口扫描 + 权重共享"的方式高效处理网格化数据（如图像）的神经网络。
---

## 创建 HelloWord.py，然后运行
stop_reason: tool_use
type: tool_use
tool: bash
input: {'command': 'cat > /Users/sunq/suibe_courses/modern_programming/books/06_Agent/HelloWord.py << \'EOF\'\nprint("Hello, World!")\nEOF'}
id: call_b6519fd968364eb4bf6260f3
---


### 构造 tool_result消息

模型返回 `tool_use` 后，程序要执行对应工具，并把结果包装成 `tool_result`。

`tool_use_id` 必须对应前面 `tool_use` block 的 `id`，这样模型才知道这段输出属于哪次工具调用。

In [14]:
tool_use_blocks = [block for block in py_response.content if block.type == "tool_use"]
if not tool_use_blocks:
    raise RuntimeError("模型本次没有返回 tool_use block，可以重新运行上一个 cell。")

block = tool_use_blocks[0]
output = run_bash(block.input["command"])

tool_result = {
    "type": "tool_result",
    "tool_use_id": block.id,
    "content": output,
}

tool_result

{'type': 'tool_result',
 'tool_use_id': 'call_b6519fd968364eb4bf6260f3',
 'content': '(no output)'}

`tool_result` 会作为新的 `user` 消息放回 `messages`。这一步完成后，模型就能基于真实命令输出继续回答。

## 5. Agent 循环

把前面的步骤合起来，就是 `agent_loop`。为了让 notebook 可以从头到尾执行，这里设置较小的 `max_steps`。

In [9]:
def agent_loop(messages: list, max_steps: int = 5):
    for _ in range(max_steps):
        response = client.messages.create(
            model=MODEL,
            system=SYSTEM,
            messages=messages,
            tools=TOOLS,
            max_tokens=8000,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return response

        results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"$ {block.input['command']}")
                output = run_bash(block.input["command"])
                print(output[:500])
                results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": output,
                })
        messages.append({"role": "user", "content": results})

    raise RuntimeError("达到最大步数，Agent 仍未结束。")

这段代码就是最小 Agent 的核心：

- 模型返回 `tool_use`，循环继续；
- 程序执行工具并生成 `tool_result`；
- `tool_result` 被放回上下文；
- 模型不再调用工具时，循环结束。

## 6. 运行一个最小任务

下面的代码会真实调用模型，并允许它运行 Bash。为了方便课堂演示，任务限定为：创建 HelloWord.py，然后运行。

In [13]:
messages = [{"role": "user", "content": "创建 HelloWord.py，内容是打印\"Hello, World!\"，然后运行它。"}]
response = agent_loop(messages)

for block in response.content:
    if getattr(block, "type", None) == "text":
        print(block.text)

$ printf 'print("Hello, World!")\n' > /Users/sunq/suibe_courses/modern_programming/books/06_Agent/HelloWord.py
(no output)
$ cd /Users/sunq/suibe_courses/modern_programming/books/06_Agent && python3 HelloWord.py
Hello, World!
已创建 `HelloWord.py` 并成功运行，输出结果为：

```
Hello, World!
```


## 7. 保存并查看完整消息历史

`messages` 中保存了用户请求、模型返回的 block、工具执行结果和最终回答。打印完整消息，可以看到 Agent 的工作过程。

以创建并运行 `HelloWord.py` 为例，常见流程如下：

1. Agent 发送用户消息，请求创建并运行 `HelloWord.py`；
2. LLM 返回 `tool_use`，生成用于创建文件的 Bash 命令；
3. Agent 执行 Bash 命令，并把结果作为 `tool_result` 返回；
4. LLM 再次返回 `tool_use`，生成运行 Python 文件的命令；
5. Agent 执行命令，并把运行结果作为 `tool_result` 返回；
6. LLM 不再调用工具，返回最终运行结果说明。

参照 `02_agent.py`，这里把完整历史保存为messages，再打印出来。

In [12]:
print(json.dumps(messages, ensure_ascii=False, indent=2, default=str))

[
  {
    "role": "user",
    "content": "创建 HelloWord.py，内容是 print(\"Hello, World!\")，然后运行它。"
  },
  {
    "role": "assistant",
    "content": [
      "ToolUseBlock(id='call_668844fa9a374c71a7f92f7a', caller=None, input={'command': 'cat > /Users/sunq/suibe_courses/modern_programming/books/06_Agent/HelloWord.py << \\'EOF\\'\\nprint(\"Hello, World!\")\\nEOF'}, name='bash', type='tool_use')"
    ]
  },
  {
    "role": "user",
    "content": [
      {
        "type": "tool_result",
        "tool_use_id": "call_668844fa9a374c71a7f92f7a",
        "content": "(no output)"
      }
    ]
  },
  {
    "role": "assistant",
    "content": [
      "ToolUseBlock(id='call_ab130b45ec444e818d0ef204', caller=None, input={'command': 'cd /Users/sunq/suibe_courses/modern_programming/books/06_Agent && python3 HelloWord.py'}, name='bash', type='tool_use')"
    ]
  },
  {
    "role": "user",
    "content": [
      {
        "type": "tool_result",
        "tool_use_id": "call_ab130b45ec444e818d0ef204",
      

也可以在终端中运行脚本：

```bash
python3 02_agent.py
```

可以尝试这些问题：

```text
当前文件夹有哪些文件
创建一个 hello.py，内容是 print("hi")，然后运行它
当前 git 分支是什么
```

观察重点：模型什么时候调用 `bash`，什么时候停止调用工具并给出最终回答。

## 小结

第一个 Agent 的关键不是工具很多，而是循环成立：

- 用户提出任务；
- 模型决定是否调用工具；
- 程序执行工具；
- 工具结果返回给模型；
- 模型继续行动，直到任务结束。

后续章节会增加更多工具、安全边界、Todo 规划和上下文管理，但核心循环仍然是这一段。